In [1]:
!pip install -q --upgrade wandb

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
import wandb
import random

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_api")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 73.5 MB/s eta 0:00:00:00:0100:01
Device: cuda


In [2]:
config = {
    "data_dir": Path("/kaggle/input/competitions/jaguar-re-id"),
    "checkpoint_dir": Path("checkpoints"),
    "megadescriptor_model": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "val_split": 0.2,
    "seed": SEED,
}

config["checkpoint_dir"].mkdir(exist_ok=True)

train_df = pd.read_csv(config["data_dir"] / "train.csv")
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['ground_truth'])
num_classes = len(label_encoder.classes_)

train_data, val_data = train_test_split(
    train_df,
    test_size=config["val_split"],
    random_state=config["seed"],
    stratify=train_df['ground_truth']
)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Classes: {num_classes}")

Train: 1516 | Val: 379 | Classes: 31


In [3]:
preprocess = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

@torch.no_grad()
def extract_embeddings(model, image_paths, batch_size=32, desc="Extracting"):
    model.eval()
    embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc=desc):
        batch_paths = image_paths[i:i+batch_size]
        tensors = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                tensors.append(preprocess(img))
            except:
                tensors.append(torch.zeros(3, config["input_size"],
                                           config["input_size"]))
        batch = torch.stack(tensors).to(device)
        embeddings.append(model(batch).cpu().numpy())
    return np.vstack(embeddings)


print("Loading MegaDescriptor...")
megadescriptor = timm.create_model(
    config["megadescriptor_model"], pretrained=True)
megadescriptor.eval()
megadescriptor.to(device)

with torch.no_grad():
    dummy = torch.randn(1, 3, config["input_size"],
                        config["input_size"]).to(device)
    megadescriptor_dim = megadescriptor(dummy).shape[1]

print(f"MegaDescriptor loaded | Dim: {megadescriptor_dim}")

cache_dir = Path("embeddings")
cache_dir.mkdir(exist_ok=True)

train_cache = cache_dir / "train_embeddings.npz"
val_cache = cache_dir / "val_embeddings.npz"

if train_cache.exists():
    train_embeddings = np.load(train_cache)["embeddings"]
    print(f"Loaded train embeddings: {train_embeddings.shape}")
else:
    print("Extracting train embeddings...")
    train_paths = [config["data_dir"] / "train/train" / f
                   for f in train_data['filename'].values]
    train_embeddings = extract_embeddings(megadescriptor, train_paths,
                                          desc="Train")
    np.savez_compressed(train_cache, embeddings=train_embeddings)
    print(f"Saved: {train_embeddings.shape}")

if val_cache.exists():
    val_embeddings = np.load(val_cache)["embeddings"]
    print(f"Loaded val embeddings: {val_embeddings.shape}")
else:
    print("Extracting val embeddings...")
    val_paths = [config["data_dir"] / "train/train" / f
                 for f in val_data['filename'].values]
    val_embeddings = extract_embeddings(megadescriptor, val_paths,
                                        desc="Val")
    np.savez_compressed(val_cache, embeddings=val_embeddings)
    print(f"Saved: {val_embeddings.shape}")

Loading MegaDescriptor...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

MegaDescriptor loaded | Dim: 1536
Extracting train embeddings...


Train:   0%|          | 0/48 [00:00<?, ?it/s]

Saved: (1516, 1536)
Extracting val embeddings...


Val:   0%|          | 0/12 [00:00<?, ?it/s]

Saved: (379, 1536)


In [4]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


class EmbeddingProjection(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=256, dropout=0.3):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim),
        )

    def forward(self, x):
        return self.network(x)

    def get_embeddings(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)


class CombinedLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, 
                 arcface_margin=0.5, arcface_scale=64.0,
                 triplet_margin=0.3, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.arcface_weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.arcface_weight)
        self.arcface_margin = arcface_margin
        self.arcface_scale = arcface_scale
        self.triplet_margin = triplet_margin

    def arcface_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.arcface_weight, dim=1)
        cosine = F.linear(embeddings, weight).clamp(-1+1e-6, 1-1e-6)
        theta = torch.acos(cosine)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.arcface_scale * torch.cos(
            theta + one_hot * self.arcface_margin)
        return F.cross_entropy(output, labels)

    def triplet_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        dist = torch.cdist(embeddings, embeddings, p=2)
        pos_mask = labels.unsqueeze(0) == labels.unsqueeze(1)
        neg_mask = ~pos_mask
        pos_mask.fill_diagonal_(False)
        hardest_pos = (dist * pos_mask.float()).max(dim=1)[0]
        hardest_neg = (dist + 1e6 * (~neg_mask).float()).min(dim=1)[0]
        return F.relu(hardest_pos - hardest_neg + self.triplet_margin).mean()

    def forward(self, embeddings, labels):
        return (self.alpha * self.arcface_loss(embeddings, labels) +
                (1 - self.alpha) * self.triplet_loss(embeddings, labels))


def compute_validation_map(model, val_embeddings, val_labels):
    model.eval()
    with torch.no_grad():
        val_tensor = torch.FloatTensor(val_embeddings).to(device)
        finetuned_emb = model.get_embeddings(val_tensor).cpu().numpy()

    sim_matrix = cosine_similarity(finetuned_emb)
    np.fill_diagonal(sim_matrix, -1)

    identity_aps = {}
    for query_idx in range(len(val_labels)):
        query_label = val_labels[query_idx]
        similarities = sim_matrix[query_idx]
        is_match = (val_labels == query_label).astype(int)
        is_match[query_idx] = 0
        sorted_indices = np.argsort(-similarities)
        sorted_matches = is_match[sorted_indices]
        n_positives = sorted_matches.sum()
        if n_positives == 0:
            continue
        cumsum = np.cumsum(sorted_matches)
        precision_at_k = cumsum / np.arange(1, len(sorted_matches) + 1)
        ap = np.sum(precision_at_k * sorted_matches) / n_positives
        if query_label not in identity_aps:
            identity_aps[query_label] = []
        identity_aps[query_label].append(ap)

    return np.mean([np.mean(aps) for aps in identity_aps.values()])


print("Model, loss and validation ready")

Model, loss and validation ready


In [5]:
sweep_config = {
    "method": "bayes",
    "metric": {"name": "val_map", "goal": "maximize"},
    "parameters": {
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 1e-3
        },
        "arcface_margin": {
            "values": [0.3, 0.4, 0.5, 0.6]
        },
        "arcface_scale": {
            "values": [32, 48, 64]
        },
        "triplet_margin": {
            "values": [0.2, 0.3, 0.4]
        },
        "alpha": {
            "values": [0.3, 0.5, 0.7]
        },
        "embedding_dim": {
            "values": [128, 256, 512]
        },
        "dropout": {
            "values": [0.1, 0.3, 0.5]
        },
    }
}


def train_sweep():
    wandb.login(key=os.environ["WANDB_API_KEY"])
    
    run = wandb.init()
    sweep_cfg = run.config

    # reset the seed
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    # datasets
    train_dataset = EmbeddingDataset(
        train_embeddings, train_data['label_encoded'].values)
    val_dataset = EmbeddingDataset(
        val_embeddings, val_data['label_encoded'].values)

    train_loader = DataLoader(train_dataset, batch_size=32,
                              shuffle=True, num_workers=0)

    val_labels = val_data['ground_truth'].values

    # model with sweep parameters
    model = EmbeddingProjection(
        input_dim=megadescriptor_dim,
        hidden_dim=512,
        output_dim=sweep_cfg.embedding_dim,
        dropout=sweep_cfg.dropout
    ).to(device)

    loss_fn = CombinedLoss(
        embedding_dim=sweep_cfg.embedding_dim,
        num_classes=num_classes,
        arcface_margin=sweep_cfg.arcface_margin,
        arcface_scale=sweep_cfg.arcface_scale,
        triplet_margin=sweep_cfg.triplet_margin,
        alpha=sweep_cfg.alpha
    ).to(device)

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(loss_fn.parameters()),
        lr=sweep_cfg.learning_rate,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)

    best_val_map = 0.0
    best_val_loss = float('inf')
    patience_counter = 0

    # training for 20 epochs / trial to save time
    for epoch in range(20):
        model.train()
        loss_fn.train()
        train_loss = 0

        for embeddings, labels in train_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            optimizer.zero_grad()
            projected = model(embeddings)
            loss = loss_fn(projected, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        # validate
        model.eval()
        loss_fn.eval()
        val_loss = 0
        val_dataset_loader = DataLoader(val_dataset, batch_size=32,
                                        shuffle=False, num_workers=0)

        with torch.no_grad():
            for embeddings, labels in val_dataset_loader:
                embeddings, labels = embeddings.to(device), labels.to(device)
                projected = model(embeddings)
                loss = loss_fn(projected, labels)
                val_loss += loss.item()

        val_loss /= len(val_dataset_loader)
        val_map = compute_validation_map(model, val_embeddings, val_labels)

        scheduler.step(val_loss)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_map": val_map,
        })

        if val_map > best_val_map:
            best_val_map = val_map
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "loss_state_dict": loss_fn.state_dict(),
                "val_map": best_val_map,
                "config": dict(sweep_cfg),
            }, config["checkpoint_dir"] / "sweep_best.pth")
        else:
            patience_counter += 1
            if patience_counter >= 5:
                break

    wandb.log({
        "best_val_map": best_val_map,
        "best_val_loss": best_val_loss,
    })

    wandb.finish()


print("Sweep config and train function ready")

Sweep config and train function ready


In [6]:
wandb.login(key=os.environ["WANDB_API_KEY"])

sweep_id = wandb.sweep(
    sweep_config,
    project="jaguar-reid-mishank"
)

print(f"Sweep ID: {sweep_id}")
print("Starting sweep agent — running 15 trials...")
print("Check your W&B project to see trials appearing live.")

wandb.agent(sweep_id, function=train_sweep, count=15)

print("\nSweep complete!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jain5 (jain5-university-of-potsdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: k2nubn59
Sweep URL: https://wandb.ai/jain5-university-of-potsdam/jaguar-reid-mishank/sweeps/k2nubn59
Sweep ID: k2nubn59
Starting sweep agent — running 15 trials...
Check your W&B project to see trials appearing live.


wandb: Agent Starting Run: 61pgfgmz with config:
wandb: 	alpha: 0.3
wandb: 	arcface_margin: 0.4
wandb: 	arcface_scale: 48
wandb: 	dropout: 0.3
wandb: 	embedding_dim: 128
wandb: 	learning_rate: 9.626834167130104e-05
wandb: 	triplet_margin: 0.3
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


best_val_loss,▁
best_val_map,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▃▄▅▅▆▇▇▇▇▇█████████
best_val_loss,0.90477
best_val_map,0.78918
epoch,20
train_loss,0.0616
val_loss,0.91964


wandb: Agent Starting Run: hq2ti7ju with config:
wandb: 	alpha: 0.5
wandb: 	arcface_margin: 0.5
wandb: 	arcface_scale: 64
wandb: 	dropout: 0.1
wandb: 	embedding_dim: 128
wandb: 	learning_rate: 1.042851715268828e-05
wandb: 	triplet_margin: 0.4
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


...

wandb: Agent Starting Run: 9jqryya8 with config:
wandb: 	alpha: 0.3
wandb: 	arcface_margin: 0.6
wandb: 	arcface_scale: 48
wandb: 	dropout: 0.5
wandb: 	embedding_dim: 256
wandb: 	learning_rate: 0.0007670550202151722
wandb: 	triplet_margin: 0.4
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


best_val_loss,▁
best_val_map,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▃▄▅▆▇▇▇▇▇██▇███████
best_val_loss,1.25715
best_val_map,0.80391
epoch,20
train_loss,0.23369
val_loss,1.24358


wandb: Agent Starting Run: lcvs2mt6 with config:
wandb: 	alpha: 0.3
wandb: 	arcface_margin: 0.4
wandb: 	arcface_scale: 32
wandb: 	dropout: 0.5
wandb: 	embedding_dim: 128
wandb: 	learning_rate: 0.0005001721232409999
wandb: 	triplet_margin: 0.4
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


best_val_loss,▁
best_val_map,▁
epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▃▂▂▂▁▂▁▂▁▁▁▁▁▁▂▁▁▁
val_map,▁▃▅▆▇▇▇▇▇▇█▇▇▇▇█▇███
best_val_loss,0.68277
best_val_map,0.80188
epoch,20
train_loss,0.0521
val_loss,0.68376



Sweep complete!


In [7]:
import wandb

api = wandb.Api()

# get all runs from the sweep
sweep = api.sweep(f"jaguar-reid-mishank/{sweep_id}")
runs = sweep.runs

# finding the best run
best_run = max(runs, key=lambda r: r.summary.get("best_val_map", 0))

print(f"Best run: {best_run.name}")
print(f"Best val mAP: {best_run.summary.get('best_val_map', 0):.4f}")
print(f"\nBest hyperparameters:")
for key, value in best_run.config.items():
    print(f"  {key}: {value}")

# show all runs sorted
print(f"\nAll runs sorted by val mAP:")
run_results = []
for run in runs:
    run_results.append({
        "name": run.name,
        "val_map": round(run.summary.get("best_val_map", 0), 4),
        "lr": round(run.config.get("learning_rate", 0), 6),
        "embedding_dim": run.config.get("embedding_dim"),
        "arcface_margin": run.config.get("arcface_margin"),
        "dropout": run.config.get("dropout"),
        "alpha": run.config.get("alpha"),
    })

results_df = pd.DataFrame(run_results).sort_values(
    "val_map", ascending=False)
print(results_df.to_string(index=False))

Best run: resilient-sweep-14
Best val mAP: 0.8039

Best hyperparameters:
  alpha: 0.3
  dropout: 0.5
  arcface_scale: 48
  embedding_dim: 256
  learning_rate: 0.0007670550202151722
  arcface_margin: 0.6
  triplet_margin: 0.4

All runs sorted by val mAP:
              name  val_map       lr  embedding_dim  arcface_margin  dropout  alpha
resilient-sweep-14   0.8039 0.000767            256             0.6      0.5    0.3
    laced-sweep-15   0.8019 0.000500            128             0.4      0.5    0.3
    clear-sweep-13   0.8008 0.000658            128             0.5      0.3    0.3
   morning-sweep-7   0.8007 0.000362            256             0.4      0.5    0.3
    worthy-sweep-8   0.8002 0.000394            256             0.4      0.5    0.3
   dainty-sweep-11   0.7953 0.000424            512             0.6      0.3    0.3
    royal-sweep-10   0.7941 0.000894            128             0.4      0.5    0.3
     swept-sweep-9   0.7929 0.000217            256             0.4      0

In [8]:
best_config = {
    "learning_rate": 0.0007670550202151722,
    "arcface_margin": 0.6,
    "arcface_scale": 48,
    "triplet_margin": 0.4,
    "alpha": 0.3,
    "embedding_dim": 256,
    "dropout": 0.5,
}

print("Training best config for 25 epochs...")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

train_dataset = EmbeddingDataset(
    train_embeddings, train_data['label_encoded'].values)
val_dataset = EmbeddingDataset(
    val_embeddings, val_data['label_encoded'].values)

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32,
                        shuffle=False, num_workers=0)

val_labels = val_data['ground_truth'].values

model = EmbeddingProjection(
    input_dim=megadescriptor_dim,
    hidden_dim=512,
    output_dim=best_config["embedding_dim"],
    dropout=best_config["dropout"]
).to(device)

loss_fn = CombinedLoss(
    embedding_dim=best_config["embedding_dim"],
    num_classes=num_classes,
    arcface_margin=best_config["arcface_margin"],
    arcface_scale=best_config["arcface_scale"],
    triplet_margin=best_config["triplet_margin"],
    alpha=best_config["alpha"]
).to(device)

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(loss_fn.parameters()),
    lr=best_config["learning_rate"],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3)

# W&B run for final training
wandb.login(key=os.environ["WANDB_API_KEY"])
wandb.init(
    project="jaguar-reid-mishank",
    name="sweep-best-final-training",
    config={**best_config, "num_epochs": 25, "num_classes": num_classes}
)

best_val_map = 0.0
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(25):
    model.train()
    loss_fn.train()
    train_loss = 0

    for embeddings, labels in tqdm(train_loader,
                                   desc=f"Epoch {epoch+1}",
                                   leave=False):
        embeddings, labels = embeddings.to(device), labels.to(device)
        optimizer.zero_grad()
        projected = model(embeddings)
        loss = loss_fn(projected, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    loss_fn.eval()
    val_loss = 0

    with torch.no_grad():
        for embeddings, labels in val_loader:
            embeddings, labels = embeddings.to(device), labels.to(device)
            projected = model(embeddings)
            loss = loss_fn(projected, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)
    val_map = compute_validation_map(model, val_embeddings, val_labels)
    scheduler.step(val_loss)

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_map": val_map,
    })

    print(f"Epoch {epoch+1:2d} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val mAP: {val_map:.4f}")

    if val_map > best_val_map:
        best_val_map = val_map
        best_val_loss = val_loss
        patience_counter = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "val_map": best_val_map,
        }, config["checkpoint_dir"] / "sweep_best_final.pth")
    else:
        patience_counter += 1
        if patience_counter >= 7:
            print(f"Early stopping at epoch {epoch+1}")
            break

wandb.log({"best_val_map": best_val_map})
wandb.finish()

print(f"\nBest val mAP: {best_val_map:.4f}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Training best config for 25 epochs...


Epoch 1:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch  1 | Train Loss: 7.1627 | Val Loss: 4.0966 | Val mAP: 0.5223


Epoch 2:   0%|          | 0/48 [00:00<?, ?it/s]

...

Epoch 24 | Train Loss: 0.2243 | Val Loss: 1.3565 | Val mAP: 0.7962


Epoch 25:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 25 | Train Loss: 0.1618 | Val Loss: 1.2961 | Val mAP: 0.8039


best_val_map,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train_loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▃▄▅▆▇▇▇▇▇██▇████████████
best_val_map,0.80391
epoch,25
train_loss,0.16179
val_loss,1.29607
val_map,0.8039



Best val mAP: 0.8039


In [10]:
# extract test embeddings
test_pairs_df = pd.read_csv(Path("/kaggle/input/competitions/jaguar-re-id") / "test.csv")
test_images = sorted(list(
    set(test_pairs_df['query_image'].unique()) |
    set(test_pairs_df['gallery_image'].unique())
))

test_cache = Path("embeddings") / "test_embeddings.npz"

if test_cache.exists():
    test_mega_embeddings = np.load(test_cache)["embeddings"]
    print(f"Loaded test embeddings: {test_mega_embeddings.shape}")
else:
    print("Extracting test embeddings...")
    test_paths = [Path("/kaggle/input/competitions/jaguar-re-id") / "test/test" / f
                  for f in test_images]
    test_mega_embeddings = extract_embeddings(megadescriptor, test_paths,
                                              desc="Test")
    np.savez_compressed(test_cache, embeddings=test_mega_embeddings)
    print(f"Saved: {test_mega_embeddings.shape}")

# project test embeddings through best model
model.eval()
with torch.no_grad():
    test_tensor = torch.FloatTensor(test_mega_embeddings).to(device)
    test_projected = model.get_embeddings(test_tensor).cpu().numpy()

print(f"Test projected: {test_projected.shape}")

# apply re-ranking with best params from Experiment 4
def k_reciprocal_reranking(query_features, gallery_features,
                            k1=15, k2=6, lambda_value=0.5):
    all_features = np.concatenate([query_features, gallery_features], axis=0)
    n_all = all_features.shape[0]
    all_features = all_features / np.linalg.norm(
        all_features, axis=1, keepdims=True)
    dist = 1 - all_features @ all_features.T
    dist = np.maximum(dist, 0)

    k_reciprocal_neighbors = []
    for i in range(n_all):
        forward_k = np.argsort(dist[i])[:k1+1]
        k_recip = []
        for j in forward_k:
            if i in np.argsort(dist[j])[:k1+1]:
                k_recip.append(j)
        k_reciprocal_neighbors.append(np.array(k_recip))

    jaccard_dist = np.zeros((n_all, n_all))
    for i in range(n_all):
        for j in range(n_all):
            set_i = set(k_reciprocal_neighbors[i])
            set_j = set(k_reciprocal_neighbors[j])
            intersection = len(set_i & set_j)
            union = len(set_i | set_j)
            jaccard_dist[i][j] = 1 - intersection / (union + 1e-6)

    final_dist = lambda_value * dist + (1 - lambda_value) * jaccard_dist
    return final_dist

print("Applying re-ranking...")
test_dist = k_reciprocal_reranking(test_projected, test_projected,
                                    k1=15, k2=6, lambda_value=0.5)
test_sim = np.clip(1 - test_dist, 0.0, 1.0)

# generate the submission
img_to_idx = {fn: idx for idx, fn in enumerate(test_images)}

similarities = []
for _, row in tqdm(test_pairs_df.iterrows(), total=len(test_pairs_df)):
    q_idx = img_to_idx[row['query_image']]
    g_idx = img_to_idx[row['gallery_image']]
    similarities.append(float(test_sim[q_idx, g_idx]))

similarities = np.clip(similarities, 0.0, 1.0)
submission_df = pd.DataFrame({
    'row_id': test_pairs_df['row_id'],
    'similarity': similarities
})

submission_df.to_csv("submission_sweep_reranked.csv", index=False)
print(f"Submission saved: {len(submission_df)} rows")

from IPython.display import FileLink
FileLink('submission_sweep_reranked.csv')

Extracting test embeddings...


Test:   0%|          | 0/12 [00:00<?, ?it/s]

Saved: (371, 1536)
Test projected: (371, 256)
Applying re-ranking...


  0%|          | 0/137270 [00:00<?, ?it/s]

Submission saved: 137270 rows


/kaggle/working/submission_sweep_reranked.csv